# Stage1 分钟现金部署模型只读失败归因

复用封存 Qlib 标签、模型预测、Qlib 分钟价格和逐层点时原生账户。对比首次合法买入、状态/普通收益控制、候选模型与不重拟合的单字段置零；核对触发后额外 SOXX 股数的价格损益、费用与复位残差。零新拟合、零新账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import hashlib,json,warnings
from copy import deepcopy
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.ibkr_tiered import TieredFees,settle_tiered_event
from src.workflow.stage1_intraday_cash_timing_information import _minute_features
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
root=Path.cwd();card=root/"research/stage1_intraday_cash_timing_failure_audit_20261028"
raw=(card/"plan.json").read_bytes();plan=json.loads(raw)
summaries={};objects={};receipts={}
provider=json.loads((root/plan["source_cards"]["information"]/"summary.json").read_text())["provider_uri"]
qlib.init(provider_uri=provider,region="us",kernels=1)
requested={"labels":["analysis/labels"],"information":["analysis/predictions","analysis/daily","models/2021_candidate","models/2022_candidate"],"dated_parent":["analysis/native/dated_f696_0"],"covered_frame":["inputs/covered_frame"]}
for role,relative in plan["source_cards"].items():
    source=root/relative;pin=plan["source_pins"][role]
    assert hashlib.sha256((source/"summary.json").read_bytes()).hexdigest()==pin["summary_sha256"]
    assert hashlib.sha256((source/"verification_manifest.json").read_bytes()).hexdigest()==pin["manifest_sha256"]
    summaries[role]=json.loads((source/"summary.json").read_text())
    objects[role],receipts[role]=load_verified_recorder_objects(source,expected_recorder_id=pin["recorder_id"],artifacts=requested[role],expected_manifest_sha256=pin["manifest_sha256"])
labels=objects["labels"]["analysis/labels"]
labels=labels.loc[labels.funded_change].sort_values(["trade_date","decision_at"]).copy()
first=labels.groupby("trade_date",sort=False).head(1).copy()
book=objects["dated_parent"]["analysis/native/dated_f696_0"]
frame=objects["covered_frame"]["inputs/covered_frame"].reset_index(drop=True)
daily=objects["information"]["analysis/daily"]
bars=D.features(["SOXL","SOXX"],["$open","$raw_open","$close","$raw_close","$raw_volume"],start_time="2018-10-15 09:31:00",end_time="2022-12-22 16:00:00",freq="1min")
minute=_minute_features(bars)
quotes={symbol:bars.xs(symbol,level="instrument") for symbol in ("SOXL","SOXX")}

# One unchanged-model input ablation isolates the added field's direct score effect.
annual=[];selected={}
for year in plan["years"]:
    part=labels.loc[labels.trade_date.dt.year.eq(year)].copy()
    anchor=first.loc[first.trade_date.dt.year.eq(year)].copy()
    candidate=objects["information"]["analysis/predictions"][f"{year}_candidate"]
    control=objects["information"]["analysis/predictions"][f"{year}_control"]
    model=objects["information"][f"models/{year}_candidate"]
    assert part.decision_at.reset_index(drop=True).equals(candidate.decision_at)
    assert candidate.decision_at.equals(control.decision_at)
    gap=minute.soxx_vwap_gap15.reindex(pd.DatetimeIndex(part.bar_end_at)).to_numpy(float)
    first_gap=minute.soxx_vwap_gap15.reindex(pd.DatetimeIndex(anchor.bar_end_at)).to_numpy(float)
    delta=gap-part.trade_date.map(pd.Series(first_gap,index=anchor.trade_date)).to_numpy(float)
    assert np.isfinite(delta).all() and np.abs(delta[part.decision_at.isin(anchor.decision_at).to_numpy()]).max()<1e-12
    feature_index=model.features.index("soxx_vwap_gap15_delta")
    direct_effect=float(model.model.coef_[feature_index]/model.scaler.scale_[feature_index])*delta
    zeroed=candidate.score.to_numpy(float)-direct_effect
    event=pd.DataFrame({"trade_date":part.trade_date.to_numpy(),"decision_at":part.decision_at.to_numpy(),"local_advantage":part.local_advantage.to_numpy(float)})
    def first_positive(scores):
        trial=event.copy();trial["score"]=np.asarray(scores,float)
        return trial.loc[trial.score.gt(0)].groupby("trade_date",sort=False).head(1).set_index("trade_date")
    chosen=first_positive(candidate.score.to_numpy(float))
    control_chosen=first_positive(control.score.to_numpy(float))
    zeroed_chosen=first_positive(zeroed)
    selected.update({pd.Timestamp(day):part.loc[(part.trade_date.eq(day)) & (part.decision_at.eq(row.decision_at))].iloc[0]
                     for day,row in chosen.iterrows()})
    calendar=daily.loc[daily.year.eq(year)].trade_date
    def value(chosen_frame):
        return calendar.map(chosen_frame.local_advantage).fillna(0).to_numpy(float)
    def action_changed(a,b):
        return int(calendar.map(a.decision_at).fillna(pd.Timestamp("1900-01-01")).ne(
            calendar.map(b.decision_at).fillna(pd.Timestamp("1900-01-01"))).sum())
    anchor_mask=part.decision_at.isin(anchor.decision_at).to_numpy()
    first_candidate=candidate.loc[anchor_mask]
    first_control=control.loc[anchor_mask]
    annual.append({"year":year,"calendar_days":len(calendar),"first_buy_sum":float(calendar.map(anchor.set_index("trade_date").local_advantage).fillna(0).sum()),
        "candidate_sum":float(value(chosen).sum()),"control_sum":float(value(control_chosen).sum()),
        "same_model_feature_zeroed_sum":float(value(zeroed_chosen).sum()),
        "direct_feature_changed_action_days":action_changed(chosen,zeroed_chosen),
        "candidate_control_changed_action_days":action_changed(chosen,control_chosen),
        "candidate_trigger_days":len(chosen),"candidate_positive_local_days":int(chosen.local_advantage.gt(0).sum()),
        "candidate_negative_local_days":int(chosen.local_advantage.lt(0).sum()),
        "first_label_positive_days":int(anchor.local_advantage.gt(0).sum()),
        "candidate_first_score_positive_days":int(first_candidate.score.gt(0).sum()),
        "median_abs_direct_score_effect":float(np.median(np.abs(direct_effect))),
        "median_abs_candidate_control_score_gap":float(np.median(np.abs(candidate.score.to_numpy(float)-control.score.to_numpy(float)))),
        "median_abs_anchor_candidate_control_score_gap":float(np.median(np.abs(first_candidate.score.to_numpy(float)-first_control.score.to_numpy(float))))})

# Reconcile the saved local branch, then split additional-share price P&L from
# execution fees, reset costs and whole-share rounding. This is descriptive.
orders=book["orders"].sort_values("stamp")
by_day={pd.Timestamp(day):part for day,part in orders.groupby("trade_date",sort=True)}
tariff=json.loads((root/plan["source_cards"]["labels"] / "plan.json").read_text())["tariff"]
fees=TieredFees(tariff,slippage_bps=0)
economics=[];max_reconstruction_error=0.
for index,row in frame.iterrows():
    day=pd.Timestamp(row.trade_date)
    current_orders=by_day.get(day,pd.DataFrame())
    if day in selected:
        item=selected[day]
        opened=day+pd.Timedelta(hours=9,minutes=47)
        pos=book["native_positions"][opened]
        q=np.array([pos.get_stock_amount(symbol) if pos.check_stock(symbol) else 0. for symbol in ("SOXL","SOXX")],float)
        cash=float(pos.get_cash())
        local_fees=deepcopy(fees)
        for order in current_orders.itertuples(index=False):
            local_fees.commit(int(order.raw_shares),stamp=order.stamp,cap_applied=bool(order.cap_applied))
        desired=np.array([item.decision_soxl_weight,item.decision_soxx_weight+item.cash_deployment_weight,item.decision_cash_weight-item.cash_deployment_weight],float)
        mark=np.array([quotes[s].loc[item.bar_end_at,"$close"] for s in ("SOXL","SOXX")],float)
        nav=float(q@mark+cash)
        fill_index=item.fill_at+pd.Timedelta(minutes=1)
        fill=np.array([quotes[s].loc[fill_index,"$open"] for s in ("SOXL","SOXX")],float)
        raw_fill=np.array([quotes[s].loc[fill_index,"$raw_open"] for s in ("SOXL","SOXX")],float)
        next_row=frame.iloc[index+1]
        next_stamp=pd.Timestamp(next_row.trade_date)+pd.Timedelta(hours=9,minutes=47)
        next_prices=next_row[["soxl_open","soxx_open"]].to_numpy(float)
        next_raw=next_row[["raw_soxl_open","raw_soxx_open"]].to_numpy(float)
        reset=book["targets"].iloc[index+1][["soxl_weight","soxx_weight","cash_weight"]].to_numpy(float)
        baseline=settle_tiered_event(q,cash,desired=reset,adjusted_prices=next_prices,raw_prices=next_raw,stamp=next_stamp,fees=deepcopy(local_fees))
        base_after=float(baseline["quantities"]@next_prices+baseline["cash"])
        branch_fees=deepcopy(local_fees)
        traded=settle_tiered_event(q,cash,desired=desired,adjusted_prices=fill,raw_prices=raw_fill,stamp=item.fill_at,fees=branch_fees)
        restored=settle_tiered_event(traded["quantities"],traded["cash"],desired=reset,adjusted_prices=next_prices,raw_prices=next_raw,stamp=next_stamp,fees=branch_fees)
        candidate_after=float(restored["quantities"]@next_prices+restored["cash"])
        reproduced=(candidate_after-base_after)/nav
        max_reconstruction_error=max(max_reconstruction_error,abs(reproduced-float(item.local_advantage)))
        gross=float((traded["quantities"]-q)@(next_prices-fill)/nav)
        economics.append({"year":day.year,"gross_additional_share_price_pnl":gross,"fees_reset_rounding_residual":float(item.local_advantage)-gross,"total":float(item.local_advantage)})
    for order in current_orders.itertuples(index=False):
        fees.commit(int(order.raw_shares),stamp=order.stamp,cap_applied=bool(order.cap_applied))
assert max_reconstruction_error<1e-9
ed=pd.DataFrame(economics)
for record in annual:
    part=ed.loc[ed.year.eq(record["year"])]
    assert len(part)==record["candidate_trigger_days"]
    record["gross_additional_share_price_pnl_sum"]=float(part.gross_additional_share_price_pnl.sum())
    record["fees_reset_rounding_residual_sum"]=float(part.fees_reset_rounding_residual.sum())
    assert abs(record["candidate_sum"]-float(part.total.sum()))<1e-12
result={"experiment":plan["experiment"],"plan_sha256":hashlib.sha256(raw).hexdigest(),
    "source_recorder_ids":{role:pin["recorder_id"] for role,pin in plan["source_pins"].items()},
    "source_readback_entries":{role:len(receipt) for role,receipt in receipts.items()},
    "annual":annual,"selected_branch_max_reconstruction_error":max_reconstruction_error,
    "new_model_fits":0,"new_native_accounts":0,"historical_research_passed":False,
    "decision":"read_only_minute_timing_failure_attribution_no_promotion"}
(card/"audit.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({"annual":annual,"selected_branch_max_reconstruction_error":max_reconstruction_error},ensure_ascii=False,indent=2))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[45605:MainThread](2026-10-28 21:16:03,756) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[45605:MainThread](2026-10-28 21:16:04,157) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[45605:MainThread](2026-10-28 21:16:04,157) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


{
  "annual": [
    {
      "year": 2021,
      "calendar_days": 252,
      "first_buy_sum": -0.003933278664032853,
      "candidate_sum": -0.004068314716376813,
      "control_sum": -0.0043019304451874015,
      "same_model_feature_zeroed_sum": -0.004215003320577827,
      "direct_feature_changed_action_days": 21,
      "candidate_control_changed_action_days": 16,
      "candidate_trigger_days": 200,
      "candidate_positive_local_days": 93,
      "candidate_negative_local_days": 107,
      "first_label_positive_days": 101,
      "candidate_first_score_positive_days": 162,
      "median_abs_direct_score_effect": 1.0596745598043636e-05,
      "median_abs_candidate_control_score_gap": 3.3134686898514205e-06,
      "median_abs_anchor_candidate_control_score_gap": 2.4375866787305824e-07,
      "gross_additional_share_price_pnl_sum": -0.0036347509592381485,
      "fees_reset_rounding_residual_sum": -0.00043356375713866506
    },
    {
      "year": 2022,
      "calendar_days": 246,
      

## 中文结论

复用封存 Qlib 分钟标签、六个逐年模型、预测、原账户和原生分钟价格；零新拟合、零新账户。所选341个整股行动分支与原标签最大误差为0。2021/2022候选局部和为−0.004068/−0.004173；新增 SOXX 股数价格损益−0.003635/−0.003927，费用、复位、整股残差−0.000434/−0.000246。首次合法时点机械买入亦在两年为负。

新增分钟字段确实改变行动：同一个候选模型把该字段归零会分别改变21/56天行动；候选与单独重拟合控制的16/33天差异不能全部归于直接字段效应。基础价值首次分数在2021偏乐观、2022偏保守，后来触发仍未可靠筛出正净值。决定 `read_only_minute_timing_failure_attribution_no_promotion`；旧固定版本继续关闭，SOXX历史Goal未通过。标签与模型NAS工件收据缺失，两个本地Recorder保留。完整数字及限制见 `conclusion.md` 和 `audit.json`。